# AutoML: Automated Machine Learning

## What is AutoML?
AutoML automates the process of selecting, building, and optimizing ML pipelines from feature preprocessing to model selection to hyperparameter tuning.

## AutoML Pipeline Components
```
Raw Data
  ↓
Data Preprocessing (imputation, encoding, scaling)
  ↓
Feature Engineering (polynomial, interactions)
  ↓
Algorithm Selection (RF, XGB, SVM, NN...)
  ↓
Hyperparameter Optimization (HPO)
  ↓
Ensemble Construction
  ↓
Final Model
```

## Tools Overview
| Tool | Approach | Best For |
|------|----------|----------|
| **AutoSklearn** | Bayesian HPO + meta-learning + ensembles | Structured/tabular data |
| **H2O AutoML** | Stacked ensembles + many algorithms | Business-ready, fast |
| **TPOT** | Genetic programming | Novel pipeline discovery |
| **Optuna** | Bayesian/TPE optimization | Custom HPO |
| **Ray Tune** | Distributed HPO | Large-scale search |
| **AutoKeras** | NAS for deep learning | Neural architecture search |
| **Ludwig** | Declarative ML | Config-driven, no code |

In [1]:
# Optuna: Most flexible and widely used HPO library
# pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data = load_breast_cancer()
X, y = data.data, data.target
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Dataset: {X.shape}, Classes: {np.bincount(y)}')

Dataset: (569, 30), Classes: [212 357]


In [2]:
# Optuna: TPE (Tree-structured Parzen Estimator) Sampler
# Objective function defines the search space and what to optimize
def objective(trial):
    # Choose algorithm
    classifier_name = trial.suggest_categorical('classifier', ['rf', 'gbm', 'svm'])
    
    if classifier_name == 'rf':
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            max_depth=trial.suggest_int('max_depth', 3, 20, log=False),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            random_state=42
        )
    elif classifier_name == 'gbm':
        clf = GradientBoostingClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 300),
            learning_rate=trial.suggest_float('learning_rate', 1e-4, 1.0, log=True),
            max_depth=trial.suggest_int('max_depth', 2, 8),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            random_state=42
        )
    else:  # svm
        C = trial.suggest_float('C', 1e-3, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly'])
        clf = Pipeline([('scaler', StandardScaler()), ('svm', SVC(C=C, kernel=kernel, random_state=42))])
    
    score = cross_val_score(clf, X, y, cv=cv, scoring='accuracy', n_jobs=-1).mean()
    return score

# Create study with Bayesian (TPE) optimization
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'Best accuracy: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

Best accuracy: 0.9772
Best params: {'classifier': 'svm', 'C': 0.07090370172923167, 'kernel': 'linear'}


In [3]:
# Optuna: Multi-objective optimization
def multi_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 300)
    max_depth = trial.suggest_int('max_depth', 2, 15)
    
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    accuracy = cross_val_score(clf, X, y, cv=3, scoring='accuracy').mean()
    
    # Minimize both: 1-accuracy (maximize accuracy) AND inference time proxy (n_estimators)
    return 1 - accuracy, n_estimators

multi_study = optuna.create_study(
    directions=['minimize', 'minimize'],
    sampler=optuna.samplers.NSGAIISampler(seed=42)  # Pareto-optimal
)
multi_study.optimize(multi_objective, n_trials=20)

print(f'Pareto front trials: {len(multi_study.best_trials)}')
for t in multi_study.best_trials[:3]:
    print(f'  Accuracy={1-t.values[0]:.4f}, n_estimators={t.values[1]}, params={t.params}')

Pareto front trials: 3
  Accuracy=0.9578, n_estimators=26.0, params={'n_estimators': 26, 'max_depth': 14}
  Accuracy=0.9561, n_estimators=15.0, params={'n_estimators': 15, 'max_depth': 15}
  Accuracy=0.9578, n_estimators=68.0, params={'n_estimators': 68, 'max_depth': 9}


## Optuna Pruning with Hyperband/ASHA

Pruning stops unpromising trials early, saving compute.

```python
import optuna
from optuna.pruners import HyperbandPruner

def objective_with_pruning(trial):
    n_epochs = 100
    learning_rate = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    hidden_dim = trial.suggest_int('hidden_dim', 32, 256)
    
    model = build_model(hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    for epoch in range(n_epochs):
        loss = train_one_epoch(model, optimizer)
        val_loss = evaluate(model)
        
        # Report intermediate value for pruning
        trial.report(val_loss, epoch)
        
        # Prune if unpromising
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return val_loss

study = optuna.create_study(
    direction='minimize',
    pruner=HyperbandPruner(min_resource=5, max_resource=100, reduction_factor=3)
)
study.optimize(objective_with_pruning, n_trials=100)
```

## H2O AutoML

```python
import h2o
from h2o.automl import H2OAutoML

h2o.init()

# Load data
train = h2o.import_file('train.csv')
test = h2o.import_file('test.csv')

x = train.columns[:-1]  # feature columns
y = 'target'
train[y] = train[y].asfactor()  # for classification

# Run AutoML trains 20 models with stacking
aml = H2OAutoML(
    max_models=20,
    max_runtime_secs=600,
    seed=42,
    balance_classes=True,
    exclude_algos=['DeepLearning']  # optional exclusions
)
aml.train(x=x, y=y, training_frame=train)

# Leaderboard
print(aml.leaderboard.head(10))

# Best model
best_model = aml.leader
perf = best_model.model_performance(test)
print(f'AUC: {perf.auc():.4f}')
```

## TPOT Genetic Programming AutoML

```python
from tpot import TPOTClassifier

tpot = TPOTClassifier(
    generations=5,
    population_size=50,
    cv=5,
    scoring='accuracy',
    random_state=42,
    verbosity=2
)
tpot.fit(X_train, y_train)
print(f'Test accuracy: {tpot.score(X_test, y_test):.4f}')

# Export best pipeline as Python code
tpot.export('best_pipeline.py')
```

## Neural Architecture Search (DARTS)

$$\min_{\alpha} L_{val}(w^*(\alpha), \alpha)$$
$$\text{s.t. } w^*(\alpha) = \arg\min_w L_{train}(w, \alpha)$$

Where $\alpha$ are continuous architecture parameters controlling which operations to use.

```python
# AutoKeras for NAS
import autokeras as ak

clf = ak.StructuredDataClassifier(max_trials=10)
clf.fit(X_train, y_train, epochs=50)
print(f'Accuracy: {clf.evaluate(X_test, y_test)[1]:.4f}')
best_model = clf.export_model()  # export as Keras model
```

## Additional Learning Resources

### Tools
- [Optuna Docs](https://optuna.readthedocs.io/) Official reference
- [Optuna Paper](https://arxiv.org/abs/1907.10902) Preferred Bayesian optimization
- [H2O AutoML Docs](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
- [TPOT Docs](http://epistasislab.github.io/tpot/)
- [AutoKeras Docs](https://autokeras.com/)
- [Ray Tune Docs](https://docs.ray.io/en/latest/tune/index.html)

### Papers
- [AutoML: A Survey of the State-of-the-Art](https://arxiv.org/abs/1908.00709)
- [Auto-Sklearn 2.0](https://arxiv.org/abs/2007.04074)
- [DARTS: Differentiable Architecture Search](https://arxiv.org/abs/1806.09055)
- [Neural Architecture Search: A Survey](https://arxiv.org/abs/1808.05377)